# `parse_pdf` — analyse complète d'un PDF en 1 ouverture fitz

Le script `parse_pdf.py` ouvre fitz **une seule fois** et retourne 4 sorties prêtes à être consommées en aval (LLM, indexation sémantique, base de données, API) :

| Sortie | Schéma |
|---|---|
| `line_df`  | 1 ligne = 1 ligne de texte du PDF (texte, bbox, font, taille, `is_invisible` pour distinguer texte natif vs couche OCR) |
| `image_df` | 1 ligne = 1 image embarquée (bbox affichée en points PDF, dimensions intrinsèques, ratio de couverture) |
| `page_df`  | 1 ligne = 1 page (`page_type`, flags additifs, comptes, `extraction_strategy`, `tool`) |
| `doc_summary` | JSON synthèse : `source_tool`, `content_type`, `recommended_strategy`, `pages_needing_ocr`, `ocr_quality`, … |

**page_type** (mutuellement exclusif) :

| page_type | Critère |
|---|---|
| `native`              | Fonts + texte natif, pas d'image plein format |
| `native_with_image`   | Fonts + texte natif + images intégrées (logos, schémas) |
| `scanned`             | Image ≥ 85 % de la page, sans couche OCR |
| `scanned_ocr_good`    | Image plein format + couche OCR exploitable |
| `scanned_ocr_bad`     | Image plein format + couche OCR dégradée (à ré-OCR-iser) |
| `mixed`               | Image plein format ET texte natif distinct |
| `empty`               | Page quasi vide |
| `unknown`             | Aucun signal décisif |

**extraction_strategy** (routage pour le pipeline aval) : `native` (fitz direct), `ocr` (OCR obligatoire), `hybrid` (texte natif + OCR sur images), `skip` (page vide).

**Type doc global** (`content_type`) : si ≥ 95 % des pages utiles dans une seule catégorie → cette catégorie ; sinon → `mixed`.

In [1]:
# À exécuter une seule fois (installation des dépendances)
import sys
!{sys.executable} -m pip install -q pymupdf pandas


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../../..'))
from docpipeline.parsing.pdf.parse_pdf import parse_pdf

PDF = '../../../../d_actif_pro_sant2.pdf'   # ← input
result = parse_pdf(PDF)

print('=== doc_summary ===')
print(json.dumps(result['doc_summary'], indent=2, ensure_ascii=False, default=str))
print()
# print('=== page_df ===')
# print(result['page_df'][['page_num', 'page_type', 'tool', 'extraction_strategy', 'char_count', 'n_images', 'image_coverage_ratio']].to_string(index=False))
# print()
# print(f"line_df  : {len(result['line_df'])} lignes")
# print(f"image_df : {len(result['image_df'])} images")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
=== doc_summary ===
{
  "pdf_hash": "e3bd5b2532b5295dbc1afccd0ea81659eb20c655c21051e6ddbab141868d30b2",
  "file_size_bytes": 495769,
  "n_pages": 6,
  "source_category": "design_tool",
  "source_tool": "Adobe InDesign",
  "source_confidence": 0.95,
  "source_signals": [
    "creator~adobe indesign",
    "xmp_history~adobe indesign"
  ],
  "creator_raw": "Adobe InDesign CS5 (7.0)",
  "producer_raw": "Adobe PDF Library 9.9",
  "pdf_version": "1.6",
  "creation_date": "D:20121109104720+01'00'",
  "modification_date": "D:20121109104839+01'00'",
  "content_type": "mixed",
  "is_scanned": false,
  "has_text_layer": true,
  "ocr_quality": "good",
  "ocr_quality_score": 1.0,
  "page_type_counts": {
    "scanned_ocr_good": 1,
    "native_with_image": 4,
    "native": 1
  },
  "scanned_page_ratio": 0.167,
  "native_page_ratio": 0.833,
  "is_encrypted": false,
  "has_form_fields": false,
  "n_vector_tables": 0,